# <center>Harness Engineering 驾驭工程 ：原理与概念</center>



&emsp;&emsp;今天我们聚焦的，是 Claude Code / Codex / Cursor 这类产品背后那套共同的工程系统——**驾驭工程（Harness Engineering）**。它是 Agent = Model + Harness 公式里的右半部分，也是 2025-2026 年 AI 工程社群共同识别出的新学科。你在聊到这些"看起来就是在调用大模型"的产品时，心里那种说不清道不明的疑问——明明底层都是 Anthropic 或 OpenAI 的模型，为什么有的能持续跑 30 分钟不失败、有的三轮就失控？为什么同样是 Sonnet 4.6，同一个人用裸 API 和用 Claude Code 的产出质量差了一个数量级？——答案全部收敛到这套系统上。

&emsp;&emsp;结合现状与调研结论，我们会重点抓住四个硬数据事件：2025 年 9 月 Vivek Trivedy（独立 AI 博客作者）给这一层第一次起名 "Agent Harness"；2025 年 11 月 Anthropic 为它写了第一篇官方系统化博文；2026 年 2 月 LangChain 用硬数据证明"只改 harness 不动模型，Terminal Bench 2.0（Stanford 推出的终端任务基准）+13.7pp（pp 即 percentage point，百分点）"；同月 OpenAI 用一个真实内测产品证明"5 个月、约 100 万行代码（on the order of）、零人工编码"。这四条数据共同定型了驾驭工程作为独立学科的地位。

&emsp;&emsp;为了把这些内容讲清楚，接下来我们按"破→立→实"三幕的主线展开。破：你会先见到三层次能力对比、同任务三模式演示、一个 50 行 naive agent 的八种失效形态，把你对"Agent 就是 LLM + 工具调用"的直觉彻底打碎。立：你会拿到八大机制 + 三支柱正交视角构成的坐标系，一张能解释所有现存 Agent 产品的"地图"。实：你会把这张坐标系套到 10+ 个主流 Harness 产品上做一次归类练习，再用 Anthropic 官方的三句警示收束课程，提醒你——驾驭工程不是银弹，它只是一套**会过时、必须准备随时删除**的脚手架。

&emsp;&emsp;你只要跟住这条主线，120 分钟之后，你至少能回答这三个问题：第一，别人说"这不就是 feedback control 吗"时，你有没有能摆出来的硬数据反击；第二，看到一个新的 Agent 产品，你能不能 3 分钟内说出它解了八大故障模式里的哪几条、属于三支柱里的哪一柱；第三，面对"2026 到底要不要学 Harness"的焦虑，你有没有一个基于时间轴和实证数据的判断依据。

> 📅 **时效性说明**：本课全部数据与引用截止 2026 年 4 月。所有官方博客 URL、博文日期、作者署名、性能数字均经过 Phase 1.5 事实核查，核查状态标注于各引用处。

> 📌 **目标受众**：本课面向**具备 Python 基础 + 有一定 Agent / LangChain 使用经验**的中级工程师。如果你正在评估"Agent 智能体开发实战课程"是否适合你，或者已学过前置课程需要系统化 Harness Engineering 知识——本课是你的入口。

> 📌 **前置知识自检**：本课是 Harness Engineering 系列第一节，主讲原理与概念，不要求你现场运行代码。但为了让你能读懂课中的示意代码与术语，你在进入本课前应当具备以下 4 项基础：
> 1. **Python 中级**：能读懂 `while/for` 循环、`try/except` 异常处理、函数定义与 docstring；
> 2. **OpenAI API 基本调用**：用过 `client.chat.completions.create(...)` 这类调用，理解 `messages` 列表与 `role` 概念；
> 3. **Function Calling 概念**：听说过"让 LLM 输出结构化 JSON 以调用工具"的思路，知道 `tools` / `tool_calls` 字段大致是做什么用的；
> 4. **LangChain 基础**：见过 `chain` 或 `agent` 这两个术语，不要求写过复杂项目，但要知道 LangChain 是用来编排 LLM + 工具的开发框架。
> 如果你对以上任一条感到陌生，建议先补齐——本课不会专门解释这些前置概念，但它们会反复在示意代码和叙述中出现。

> 📌 **前置术语小词典**（以下术语会在正文反复出现，首次遇见时正文还会有括注提示；读正文卡壳时回来查一下即可）：
> - **pp**：percentage point，百分点。例 "+13.7pp" 即"提升 13.7 个百分点"，不是百分之十三点七。
> - **Terminal Bench 2.0**：Stanford 与合作方推出的终端任务基准，考核 Agent 在真实 shell 环境完成复杂任务的通过率。
> - **SWE-bench / SWE-bench Verified**：Princeton 提出的 GitHub issue 修复基准；Verified 是经人工复核的高质量子集，是 2025 年后业界公认的"Agent 能不能真修 bug"的黄金标尺。
> - **prompt cache**：LLM 按输入前缀命中的缓存机制。前缀不变可复用，让多轮调用的延迟和费用大幅下降；一旦前缀被改写，缓存全部作废。
> - **HaaS**：Harness-as-a-Service，把 harness 封装为可订阅服务的形态，由 Vivek Trivedy 在 2025-09-23 首篇博文中创造。
> - **GAN**：Generative Adversarial Network，生成对抗网络，核心思路是"生成者 vs 鉴别者"两模型对抗提升质量；本课 6.8 节借用其思路做架构设计，不做 ML 训练实现。
> - **Bitter Lesson**：Rich Sutton 2019 年的 AI 经典论文，惨痛教训——简单通用方法最终胜过复杂领域知识。本课第四章 Phil Schmid 提出的"三原则"就是对这一理念的致敬。

---

## <center>第一章:驾驭工程是什么</center>

&emsp;&emsp;本节从一个可能让你不太舒服的冲击开始。你大概率已经用过 ChatGPT 写代码、用过带 Code Interpreter 的 ChatGPT 让它真的"跑"一段代码、也见过 Claude Code 这种能在你电脑上改十几个文件、运行测试、自己调试的工具。如果被问"这三者最大的差别在哪里"，我们很多人的第一反应会是什么？大概率会说"模型能力越来越强"，或者"Claude Code 用的是更贵的模型"。这个直觉是一个典型误解——典型到 B 站评论区里最高赞的三条评论都在重复这个错误。

&emsp;&emsp;本节的第一件事，就是把这个直觉打碎。我们会先用一张三层次能力对比表看清差异到底在哪里，再用一段 Agent = Model + Harness 的公式把这个差异翻译成工程语言，最后用一个 F1 赛车的类比把这个公式钉进你的长期记忆。

### 1.1 三层次能力对比：差异从来不在生成代码，在持续行动

&emsp;&emsp;我们先看下面这张表。它把三种"你多半都用过"的产品放在同一张纸上，用六个维度逐条比对。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 1-1 三层次能力对比</font></p>
<div class="center">

| 维度 | ChatGPT（裸对话） | ChatGPT + Code Interpreter | Claude Code |
|------|------------------|--------------------------|-------------|
| 能写代码吗 | 能 | 能 | 能 |
| 能真跑代码吗 | 否 | 能（沙箱内） | 能（你本地） |
| 能读写你的文件系统吗 | 否 | 否（只能沙箱临时文件） | 能（整个项目目录） |
| 能跨工具编排吗（git/pytest/shell） | 否 | 部分（Python 内） | 能 |
| 能持续 30 分钟不失败吗 | 否 | 否（context 溢出） | 能（机制保护） |
| 任务失败时会自己补救吗 | 否 | 部分（在同一回合内） | 能（跨回合 + 跨 session） |

</div>

&emsp;&emsp;看到这张表，我们要注意到一个反差：**前两行三列几乎没有差距**（都能写代码、都能在沙箱里跑），但从**第 3 行开始三列几乎跨了三个时代**。而三列用的模型——GPT-5 / GPT-5 / Sonnet 4.6——**都是当前最前沿的旗舰**，能力基础在同一梯队。既然模型本身差距不大，造成第 3 行之后巨大落差的就只剩一个变量：模型**外面**那一层，也就是本课要讲的 harness。

&emsp;&emsp;换个更狠的说法：把 Claude Code 里的 Sonnet 4.6 换成 GPT-5，它还是比裸的 GPT-5 强；把裸 ChatGPT 的模型升级到 Sonnet 4.6，它也追不上最朴素的 Claude Code。为什么？因为一次裸 API 调用产生的那次对话本身，就没有文件系统、没有 git commit 能力、没有跨 session 的进度文件——这些差异不在模型权重里，在**调用模型的那套系统**里。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260423171849435.svg" width="70%">
</div>

### 1.2 Agent = Model + Harness：本课第一个公式

&emsp;&emsp;这个"调用模型的那套系统"就是 Harness。把它写成公式：

> **Agent = Model + Harness**
> —— Vivek Trivedy（独立 AI 博客作者，HaaS / Harness-as-a-Service 术语创造者）在[首篇 HaaS 博文](https://vtrivedy.com/posts/claude-code-sdk-haas-harness-as-a-service)（2025-09-23）中给出这一公式化表达；Birgitta Böckeler（Thoughtworks 作者，martinfowler.com 投稿）于 2026-04-02 在 [Harness engineering for coding agent users](https://martinfowler.com/articles/harness-engineering.html) 整理传播

&emsp;&emsp;这个公式不是标新立异，它是 2026 年一季度整个 AI 工程社群共同凝练出来的结论。我们可以把它类比成 F1 赛车：**Model 是引擎，Harness 是底盘、变速箱、空气动力学、刹车、方向盘、刹车盘油路**。引擎再强，没有底盘和刹车，跑不过 100 公里就失败；引擎差一点，但底盘调校好了，能在一个赛季跑满二十几个大奖赛。

&emsp;&emsp;你现在可能还觉得这是个"比喻生动但证据不足"的公式。没关系，接下来的 120 分钟里，我们每一章都会把这个公式的右半部分拆开——Harness 到底由什么组成、为什么少了会失败、怎么装上去才能跑。但在进入下一节之前，可以先记住一个数字作为"公式的第一块硬证据"：**同模型只改 harness，Terminal Bench 2.0 从 52.8% → 66.5%，+13.7 个百分点**。这条数据会在第二章详细展开——它就是 Agent = Model + Harness 公式的第一层验证层，让你在记住公式的同时已经有可核验的实证材料。

### 1.3 同任务三模式：让差异从表格里跳出来

&emsp;&emsp;本节结尾，我们要在脑子里留下三个画面（以下三个画面是三种模式在同一任务上的真实对比，你可以边看边想象操作差异）：

- **画面 A（裸 API）**：把"帮我实现一个待办应用，包括前端 HTML 和后端 Python + pytest"的任务扔给裸 GPT-5。它会返回三块代码，但它永远不会真的创建文件、运行测试、看到测试失败后自己改。它**只是在对话框里写字**。


- **画面 B（Code Interpreter）**：同样的任务，它会真的创建 .py 文件、运行 pytest、看到失败。但它没有访问你本地项目的能力，没有 git，也没有任何跨 session 的持久化。关闭窗口，所有进度清零。


- **画面 C（Claude Code）**：同样的任务，它在你本地创建完整项目结构，写完代码跑 pytest，失败了就改，最后 commit 到 git，下次打开从 claude-progress.txt 接着跑。

&emsp;&emsp;三个画面的差距，不是"模型变聪明了"，是"模型外面被加了一层肉眼看不到的工程系统"。这就是我们今天要学的东西。

> **【学完本节你已经掌握】**
> - 用**六个维度对比表**说清楚 ChatGPT 裸对话 / Code Interpreter / Claude Code 三种产品的核心差别
> - 记住并能解释 **Agent = Model + Harness** 这个公式，知道差距的根源**不在模型而在 Harness 层**
> - 拿到公式的第一块硬证据（+13.7pp）并知道它来自 LangChain 2026-02-17 同模型对比实验

---

## <center>第二章:承认抵触——为什么驾驭工程不是新瓶装旧酒</center>

&emsp;&emsp;讲到这里，我们需要正面处理一个情绪。如果你是做过几年工程的开发者，听到"Harness Engineering"这个新词，第一反应多半是抵触。B 站顶评里有三条最高赞的评论，每一条都代表了一种典型的抵触姿态。

### 2.1 承认抵触：你的软件工程师直觉没有错

> 有人说："这不就是 feedback control 吗？控制系统搞了几十年了。"
> 有人说："换个名字就能发博客涨粉，2026 的 AI 圈越来越水。"
> 有人说："LangChain 三年前就在做这件事，只是现在包装成新概念。"

&emsp;&emsp;这三条评论都不需要立刻驳斥——它们代表的工程直觉**都是对的**。硬件工程师、嵌入式工程师、做控制理论的工程师，确实在几十年前就有 feedback control loop；LangChain 2022 年就发布了，AutoGPT 2023 年就能跑多轮工具调用；"新瓶装旧酒"是 AI 圈的常见操作。对新术语保持怀疑，是一个成熟工程师该有的态度。

&emsp;&emsp;但是，怀疑不等于否定。接下来我们要做的事，是摆出四条硬数据，让你自己判断——"Harness Engineering"到底是有工程价值的新学科，还是又一个包装过度的流行词。

### 2.2 硬数据反击：四条来自一手源的证据

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-1 Harness Engineering 硬数据反击表</font></p>
<div class="center">

| 数据 | 出处 | 含义 |
|------|------|------|
| 同模型仅改 harness，Terminal Bench 2.0 从 52.8% → 66.5%，**+13.7pp**，排名 Top 30 → Top 5（测试模型：gpt-5.2-codex；同一任务集 Terminal Bench 2.0；测试时间 2026-02-17） | LangChain 博客 2026-02-17，作者 Vivek Trivedy：<br>[Improving DeepAgents with Harness Engineering](https://www.langchain.com/blog/improving-deep-agents-with-harness-engineering) | 证明 harness 是可测量的工程变量，不是玄学 |
| OpenAI 内测产品：5 个月、**约 100 万行代码**（原文 "on the order of a million"）、**约 1500 个 PR**（原文 "roughly 1,500"）、**全程零人工编码**、开发时间**约为传统方式的 1/10**（原文 "on the order of ten times faster"；博文注明此为团队内部估算，非严格对照实验） | OpenAI 工程团队博文 2026-02-11：<br>[Harness engineering: leveraging Codex in an agent-first world](https://openai.com/index/harness-engineering/) | 证明 harness 可以承担生产级工程强度 |
| Manus 团队 6 个月内重构 harness **5 次**，每次都是删掉过时的复杂逻辑 | Phil Schmid 博文 2026-01-05：<br>[The importance of Agent Harness in 2026](https://www.philschmid.de/agent-harness-2026) | 证明 harness 是活的工程系统，不是一次性架构 |
| 2026-04-02 Birgitta Böckeler 在 martinfowler.com 发表 "Harness engineering for coding agent users"，Martin Fowler 系博客正式纳入 | [martinfowler.com: Harness engineering for coding agent users](https://martinfowler.com/articles/harness-engineering.html) | 证明业界权威博客已把它当作成熟命名的工程学科 |
</div>

> status: Manus 6 次数据待核实（源：philschmid.de/agent-harness-2026，Manus 数据为 Schmid 博文引用，原始一手来源待查）

&emsp;&emsp;这四条数据里，最需要记住的是第一条和第二条。第一条告诉我们——**harness 是可以被 benchmark 量化的**，不是停留在概念层。第二条告诉我们——**harness 可以承担生产级强度**，OpenAI 敢在官方博客上说"约 100 万行（on the order of）、零人工编码"，这是 OpenAI 以官方发布形式公开的技术声明——公司官方名义公开的数字通常具备更强的可信度担保。

&emsp;&emsp;所以现在回过头看，它还是 feedback control 换皮吗？feedback control 没人给它发 +13.7pp 的 benchmark 证明，也没有公司敢说"用它写了约 100 万行（on the order of）零人工代码"。这不是换皮，是**一个已经能拉硬数据的新工程学科**。

### 2.3 译名锁定：本课程主术语用"驾驭工程"

&emsp;&emsp;最后我们锁定这门课的中文译名。英文原词是 "Agent Harness"，harness 这个词本意是"马具"——套在马身上让人驾驭它的皮带、笼头、缰绳。把它翻成中文有几种候选：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-2 译名候选对比</font></p>
<div class="center">

| 候选译名 | 优点 | 缺点 |
|---------|------|------|
| 马具工程 | 直译，保留原始隐喻 | "马具"在中文语境过于生活化，弱化了工程严肃性 |
| 装具工程 | 工程感较强 | "装具"在中文有军事/装备含义，容易引起歧义 |
| 脚手架工程 | 常见 Agent 开发说法 | scaffolding 和 harness 是**不同的概念**，学术上 scaffolding 更宽泛 |
| **驾驭工程**（本课锁定） | 保留"驾驭"这个核心动作，也保留工程学科感 | 略偏意译，但比直译更容易被中文受众接受 |

</div>

> 📌 本课程主术语锁定为 **"驾驭工程"**。从本节开始，"驾驭工程"指代 Harness Engineering，"harness"（小写原词）指代具体的一套机制组合。

---

## <center>第三章:三代工程的包含关系与术语边界</center>

&emsp;&emsp;破完情绪，我们接下来开始立框架。这一节是整门课最偏"字典式"的一节——我们要把一组肯定听过但没彻底分清的术语一次性捋清楚：Prompt Engineering、Context Engineering、Harness Engineering 之间到底是什么关系；Agent、Scaffolding、Framework、Runtime 这四个词各自和 Harness 的边界在哪里；以及 LangChain 本身是什么——是 Framework、Runtime，还是一个 Harness？

### 3.1 三代工程：包含关系而非替代关系

&emsp;&emsp;你可能已经被"上下文工程取代提示词工程"这种标题党文章荼毒过。我们先把这个误解掰正——**没有谁取代谁，只有谁包含谁**。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260423152728592.svg" width="60%">
</div>

&emsp;&emsp;三代工程不是"旧的被淘汰、新的上位"的替代关系，而是**同心圆式的包含关系**——写 prompt 的时候不会突然不需要上下文，管理 context 的时候也不会突然不需要 prompt。每一代都把前一代的能力**吸纳进来**，再加一层新的工程维度。

&emsp;&emsp;提示词工程（2022-2023）管的是**单轮 LLM 对话**里如何把问题问得更好。上下文工程（2024-2025）管的是**多轮对话里**喂给 LLM 的信息如何更精准——包括 RAG、tool calling 的 schema、对话历史管理、instruction caching。驾驭工程（2025-2026）管的是**多轮工具调用 + 长时间运行**时，整个系统如何不失败——包括循环刹车、状态持久化、验证闭环、子代理分治。

&emsp;&emsp;下次再看到"上下文工程取代提示词工程"或"驾驭工程取代上下文工程"这种标题，你可以直接关掉。作者没有理解同心圆。

&emsp;&emsp;为了让"包含而非替代"不只停留在图示，我们把它落到一个真实产品上走一遍——**以 Claude Code 处理"写登录功能"这个任务为例**：当它启动时，**Prompt Engineering 层**在工作——system prompt 里写着"你是编程助手、使用 Anthropic 推荐的指令工程范式"；同时 **Context Engineering 层**也在工作——它会读取你项目目录里的 CLAUDE.md、把已有 tool 的 schema 注入 context、监测并压缩过长的 tool 输出；同时 **Harness Engineering 层**还在工作——它用 max_iterations 硬刹车兜底循环、用 progress 文件持续保存进度、用 compaction 阈值触发 context 清理。一次 Claude Code 会话同时运行着三代工程的所有机制，没有哪一层被"替代"，只有层层叠加。这是包含关系最朴素的验证——**同一个产品可以同时被三代工程的任意一代观察，都能看到对应机制在工作**。

### 3.2 术语边界：Agent / Scaffolding / Framework / Runtime vs Harness

&emsp;&emsp;现在我们把身边几个高频术语一次性捋清楚，每个术语用"是什么 / 解决什么 / 与 Harness 区别"三连定位。在看表之前先认识两个后面反复出现的名字：**LangGraph** 是 LangChain 2024 年推出的状态机运行时（checkpoint / interrupt / 状态持久化），**DeepAgents** 是 LangChain 2025 年基于 LangGraph 预装八大机制的 harness 成品库——下一节 3.3 会展开讲，表里先见到名字不认识不用慌。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-1 术语边界对照</font></p>
<div class="center">

| 术语 | 是什么 | 解决什么 | 与 Harness 的关系 |
|------|-------|---------|------------------|
| Agent | 一个"会用工具完成任务"的软件实体（概念层） | 让 LLM 具备行动能力，不只是回答 | Agent 是**目标**，Harness 是**让目标能持续运行的工程系统** |
| Scaffolding | prompt 到达模型前的组装层（含 prompt template / few-shot 拼接 / tool schema 注入） | 让 LLM 的输入更可控 | Scaffolding 是 Harness 的**一部分**（偏 context 装配层） |
| Framework | 提供类库和抽象的开发工具包（LangChain / LlamaIndex 等） | 让你不用从零写 LLM 调用链 | Framework 是**积木**，Harness 是用积木**搭出来的整车** |
| Runtime | 负责执行 agent 逻辑的运行时引擎（LangGraph / DeepAgents SDK） | 让 agent 的状态机在工业环境可运行 | Runtime 是**发动机**，Harness 是发动机 + 底盘 + 传动 |

</div>

&emsp;&emsp;这里需要特别留意一个陷阱——**"scaffolding" 在学术文献里比 harness 宽泛很多**，甚至会覆盖到 prompt template 和微调数据。今天这门课里，我们把 scaffolding 限定在"prompt 到达模型前的组装层"这个实用定义上，不展开学术边界争议（如果你想追溯完整的学术定义，可以搜 arXiv 2603.05344）。

> **【常见误区】** 把 scaffolding 当 harness 的同义词**
> **后果**：很多博客把 "prompt scaffolding" 和 "harness" 混用，让读者低估 harness 的工程复杂度——以为做好 prompt 组装就等于做好了 harness，漏掉了 runtime 执行层的循环刹车、进度续传、验证闭环等关键机制。
> **正确做法**：按本课的三代工程同心圆图定位——scaffolding 只是 context 装配层的一个子集，是 harness 的**一部分**；真正的 harness 必须覆盖 context + runtime + 长任务保护三层。
> **排查方法**：看一个系统被宣称是 harness 时，检查它是否包含：(a) 循环终止刹车、(b) 跨 session 进度持久化、(c) 真机功能验证——三者缺一就不是完整 harness。

### 3.3 LangChain 三层架构：Framework、Runtime、Harness 同一家

&emsp;&emsp;你大概率学过 LangChain，所以我们接下来用 LangChain 这个你最熟悉的例子把上面四个术语同时落地。LangChain 创始人 Harrison Chase 在 2025-10-25 博文 [Agent Frameworks, Runtimes, and Harnesses - oh my!](https://blog.langchain.com/agent-frameworks-runtimes-and-harnesses-oh-my/) 里把产品线切成了三层。

&emsp;&emsp;在看表之前先做个小铺垫：如果你用过 LangChain 的 LCEL 或 AgentExecutor，你已经在用 Framework 层；**LangGraph** 是 LangChain 2024 年推出的**状态机引擎**，把一次对话升级为可暂停、可恢复的有状态工作流（这就是表里提到的 checkpoint / interrupt）；**DeepAgents** 则是在 LangGraph 之上预装好八大机制的"整车"。如果这三者的区别暂时记不住不用慌——我们在第八章生态地图里会再次见到它们，到时候再回过头看这张表会更清晰。顺便补一个生态定位：**LangChain 在整体 AI 技术栈中位于 Framework+Runtime+Harness 层**，上游是 LLM API（OpenAI / Anthropic / 国产模型），下游是应用（ChatBot / IDE / CLI）。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-2 LangChain 三层产品架构</font></p>
<div class="center">

| 层 | LangChain 对应产品 | 角色 | 类比 |
|---|------------------|------|------|
| Framework | langchain-core / integrations | 提供 LLM / tool / retriever 的抽象基类 | 积木 |
| Runtime | LangGraph | 提供状态机 / checkpoint / interrupt 的执行引擎 | 发动机 |
| Harness | DeepAgents | 把 LangGraph 组装成"能跑长任务"的完整系统 | 整车 |

</div>

&emsp;&emsp;看到这张表，我们就理解了为什么 2026 年之后 LangChain 的主推产品从 AgentExecutor 转向 DeepAgents——前者是 Runtime 级别的抽象（工程师还要自己拼装成 Harness），后者本身就是一个 Harness 成品。Harrison Chase 在那篇博文里说得很直白：大多数工程师需要的不是 runtime，是一个**能直接跑的 harness**，LangChain 要把 harness 这一层也做完。

&emsp;&emsp;到这里我们手里已经有了 Agent = Model + Harness 公式、三代工程包含关系、四个术语边界、LangChain 三层落地——这是本课所有后续内容的"字典基底"。接下来我们要做一件更决定性的事：把一个没有 harness 保护的 naive agent 拉出来现场复现失效，让你亲眼看到缺了 harness 的 agent 到底有多脆弱。

---

## <center>第四章:为什么此刻是驾驭工程的拐点</center>

&emsp;&emsp;你现在可能会问：Agent 不是 2023 年 AutoGPT 就有了吗？为什么驾驭工程是 2025-2026 才立成学科？这一节我们用 Karpathy 的一句话 + 一个 8 节点时间轴，把这个问题彻底回答。

### 4.1 Karpathy 时间判断：2025 Q3 是分水岭

> "Agents basically didn't work before December [2024]. Now they just barely work. And this is the window where harness engineering becomes real engineering."
> —— Andrej Karpathy（[YC AI Startup School 2025-06-17 主题演讲 "Software Is Changing (Again)"](https://www.youtube.com/watch?v=LCEmiRjPEtQ)，后续在 X/Twitter 反复引用）

&emsp;&emsp;Karpathy 这句话翻译过来是："Agent 在 2024 年 12 月之前基本上跑不起来。现在它们勉强能跑起来。而这个**勉强能跑起来**的窗口，正是驾驭工程成为真正工程学科的时间点。" 原文的关键词是"勉强（barely）"——它不是说模型变神了，而是说**模型终于跨过了某个基准线**，让"在模型外面围一层系统让它长时间运行"从不可能变成可行。

&emsp;&emsp;那个基准线具体是什么？大致可以用 SWE-bench（Princeton 提出的 GitHub issue 修复基准）60% 通过率来标——当 Claude 3.5 Sonnet 在 2025 年 Q3 把 SWE-bench Verified（经人工复核的高质量子集）带到 60% 这个量级，模型已经能"完成多数简单任务"，这时候我们在外面加 harness 才有边际收益。模型只有 30% 通过率的时候，harness 设计得再好也救不了；模型 90% 通过率的时候，harness 要做的事反而变少。**60-90% 这个区间，是驾驭工程的黄金窗口**——而这个窗口恰好从 2025 Q3 开始。

### 4.2 8 节点时间轴：术语怎么一步步被命名的

&emsp;&emsp;下面这张表，是"Agent Harness" 这个术语从首用到定型的完整历史。请注意第一行——**术语首用是 Vivek Trivedy（2025-09-23）**，不是 Harrison Chase，也不是 Anthropic。这一点很多博客写错了，本课在这里一次性正名。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-1 驾驭工程术语演化 8 节点时间轴</font></p>
<div class="center">

| # | 日期 | 事件 | 作者 / 出处 |
|---|------|------|------------|
| 1 | 2025-09-23 | 博文 ["The Claude Code SDK and the Birth of HaaS（Harness-as-a-Service，把 harness 封装为服务形态）"](https://vtrivedy.com/posts/claude-code-sdk-haas-harness-as-a-service) 首次使用 "Agent Harness" 术语 | Vivek Trivedy（独立 AI 博客作者，vtrivedy.com） |
| 2 | 2025-10-25 | 博文 [Agent Frameworks, Runtimes, and Harnesses - oh my!](https://blog.langchain.com/agent-frameworks-runtimes-and-harnesses-oh-my/) 明确引用并传播该术语，并自认 "I didn't come up with it" 并链接 Trivedy | Harrison Chase（LangChain 博客） |
| 3 | 2025-11-26 | 官方首篇系统化博文 [Effective Harnesses for Long-Running Agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) | Anthropic，作者 Justin Young |
| 4 | 2026-01-05 | 博文 [The importance of Agent Harness in 2026](https://www.philschmid.de/agent-harness-2026)，提出 Bitter Lesson（Rich Sutton 2019 经典，简单通用方法最终胜过复杂领域知识）三原则（Start Simple / Build to Delete / Harness is the Dataset） | Phil Schmid（Hugging Face 技术博主，philschmid.de） |
| 5 | 2026-02-11 | 官方博文 [Harness engineering: leveraging Codex in an agent-first world](https://openai.com/index/harness-engineering/)，用生产实证定型术语 | OpenAI 工程团队 |
| 6 | 2026-02-17 | 博文 [Improving DeepAgents with Harness Engineering](https://www.langchain.com/blog/improving-deep-agents-with-harness-engineering)，用 +13.7pp 实证证据证明 harness 的可量化价值 | Vivek Trivedy @ LangChain |
| 7 | 2026-02-18 | 博文 [A Guide to Which AI to Use in the Agentic Era](https://www.oneusefulthing.org/p/a-guide-to-which-ai-to-use-in-the)，用三分框架把概念推向大众 | Ethan Mollick（Wharton 商学院教授，AI 大众传播者，oneusefulthing.org） |
| 8 | 2026-03-24 | 博文 [Harness Design for Long-Running Application Development](https://www.anthropic.com/engineering/harness-design-long-running-apps)，提出 GAN（生成对抗网络，"生成者 vs 鉴别者"对抗思路）启发的 Planner-Generator-Evaluator 三角色架构 | Anthropic，作者 Prithvi Rajasekaran |

</div>

&emsp;&emsp;仔细看这张时间轴，我们会发现一个有意思的规律：**前 4 个节点（2025-09 到 2026-01）是社群博客驱动的概念孵化**；**后 4 个节点（2026-02 到 2026-03）是大厂用实证数据 + 官方架构博文把它钉成学科**。Trivedy 和 Schmid 是种子，Anthropic 和 OpenAI 是定调者，Mollick 是大众化传播者——六个月完成了一门学科从诞生到定型的完整周期。

&emsp;&emsp;还有一个细节：Anthropic 在 6 个月里一口气发了**两篇**官方博文（2025-11-26 和 2026-03-24），前一篇讲"机制是什么"，后一篇讲"架构怎么设计"。这个节奏说明什么？说明 Anthropic 把驾驭工程当成**严肃的工程学科**在持续投资，不是一次博客营销。

### 4.3 实证数据收束：两个决定性证据

&emsp;&emsp;回到 Karpathy 那句话的核心——"勉强能跑起来"的时代，决定成败的不再是模型选型，而是 harness 设计。两条硬数据让这个判断落地：

> **证据一（LangChain 2026-02-17）**：同模型、同 benchmark、只改 harness，Terminal Bench 2.0 从 52.8% → 66.5%，**+13.7 个百分点**。当你用这一条去反驳"这是 feedback control 换皮"的人，对方没得反驳——feedback control 没人能给你看 benchmark 曲线。

> **证据二（OpenAI 2026-02-11）**：一个内测产品，5 个月、**约 100 万行代码（原文 "on the order of a million"）**、**约 1500 个 PR（原文 "roughly 1,500"）**、全程零人工编码、开发时间**约为传统方式的 1/10**。这是 OpenAI 以官方博文形式做的技术声明，可以当作"驾驭工程能承担生产级工程强度"的硬证据。

&emsp;&emsp;**这两条证据合起来的教学结论**：决定模型能不能上线的是模型本身，决定**能不能稳定交付**的是 harness。前者是 2023 年之前 AI 工程的核心问题，后者是 2026 年及以后的核心问题——我们现在学驾驭工程，不是跟风，是踩在学科刚立起来的最佳时间点上。

---

## <center>第五章:看 naive agent 如何失效——八大故障模式清单</center>

&emsp;&emsp;这一节是整门课最具冲击力的一节。接下来你会看到一段手写的 ~50 行 naive agent——它**没有**任何 harness 保护，所有的 LLM + tool calling 就是教科书式的 while True + JSON schema。然后我们会把它放到一个非常朴素的任务里："读一个 Python 测试文件、修 bug、跑 pytest 确认通过"。这个任务在 Claude Code 里 2 分钟能做完，在这个 naive agent 里——你会看到它用三种不同的方式失效。

&emsp;&emsp;看完这三种失效形态，我们就会理解为什么后面第六章的八大机制每一个都有它的存在价值——**它们不是从天上掉下来的"最佳实践"，它们是被一次次实战教训打磨出来的工程经验**。

### 5.1 演示准备：~50 行 naive agent 完整代码

&emsp;&emsp;下面是这个 naive agent 的完整代码。我们先通读一遍，不用运行——代码里用 `# 故障 ①～⑧` 注释标出了主要失效点；其中故障 ③（cache miss）在主循环的"每轮改写 system prompt"那一行真实触发（见下文标注"故障 ③ 触发点"的注释），其他故障多以结构性缺陷形式存在。**建议的阅读顺序**：第一遍直接跳到每个函数的 docstring 和代码里的 `# 故障 ①` 注释，忽略具体实现细节；第二遍如果你对某个故障模式的触发机制感兴趣，再回去看对应函数体——这样 62 行代码的认知负担会大幅下降。

&emsp;&emsp;**代码定位**：本讲所有代码都是**骨架示意**（演示失效形态、机制形态），不追求可运行。完整的工程实现、真实 E2E 运行轨迹与 Baseline vs Full Harness 对比数据见**第二讲《手搓 Mini Harness》**——那一讲我们会把 8 大机制全部跑起来。

In [ ]:
"""
naive_agent_demo.py — 教学示意：~50 行 naive tool-calling 循环
故意保留全部 8 个故障点，不作生产参考。
运行前需设置 OPENAI_API_KEY；课堂演示建议只读不跑。
"""
import os
import json
import subprocess
from openai import OpenAI  # 依赖见 requirements.txt（openai>=1.50.0）

MODEL = os.getenv("OPENAI_MODEL", "gpt-5")  # 优先读环境变量，禁止硬编码
SAFE_MAX_ITER_FOR_CLASSROOM = 3  # 课堂保护；真正 naive agent 根本没有这个变量
client = OpenAI()

# 3 个 tool 的 JSON schema（按 OpenAI function-calling 规范声明）
tools = [
    {"type": "function", "function": {"name": "read_file",  "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}},
    {"type": "function", "function": {"name": "write_file", "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]}}},
    {"type": "function", "function": {"name": "run_pytest", "parameters": {"type": "object", "properties": {}}}},
]

# —— 3 个 tool 实现全部无保护 ——
def read_file(path):  # 故障 ④：FileNotFoundError 会被主循环 except 吞成空串
    with open(path) as f:
        return f.read()

def write_file(path, content):  # 故障 ⑥：无权限闸，可覆盖任何文件
    with open(path, "w") as f:
        f.write(content)
    return "done"

def run_pytest():  # 故障 ④：丢弃 returncode，失败也只返回 stdout，agent 看不到失败
    return subprocess.run(["pytest"], capture_output=True, text=True).stdout

# —— 主循环：八大故障模式埋点全部在这里 ——
def naive_agent(task):
    messages = [
        # 故障 ③：system prompt 嵌入会变化的 session_iter，每轮破坏前缀，prompt cache 全部 miss
        {"role": "system", "content": "你是编程助手。完成任务后停止。[session_iter=0]"},
        {"role": "user", "content": task},
    ]
    # 故障 ①：真正 naive 这里是 while True，SAFE_MAX_ITER 只是课堂保护
    for i in range(SAFE_MAX_ITER_FOR_CLASSROOM):
        messages[0]["content"] = f"你是编程助手。完成任务后停止。[session_iter={i}]"  # 故障 ③ 触发点
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg)  # 故障 ②：messages 无限 append，context 单调膨胀
        if msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                try:
                    # 故障 ⑥：无危险命令检测，任何 tool 直接执行
                    result = {"read_file": read_file, "write_file": write_file, "run_pytest": run_pytest}[tc.function.name](**args)
                except Exception:
                    result = ""  # 故障 ④：异常吞成空串，agent 以为一切正常
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            break  # 故障 ⑦：无独立 Evaluator，agent 自报完成即退出；故障 ⑤：退出即丢进度

if __name__ == "__main__":
    # 故障 ⑧：无 token/cost budget，跑多少烧多少
    naive_agent("读取 test_calculator.py，修复失败的测试，运行 pytest 确认通过")


&emsp;&emsp;这段代码是一个诚实的 "tool-calling 教科书实现"——100 行左右完成了所有"必要的事"：调 LLM、解析 tool_calls、执行工具、把结果塞回 messages、下一轮。如果你是刚学 Agent 的人，这就是你会从 OpenAI cookbook 里抄来的版本。**它的罪不在写得差，在太朴素**——朴素到完全没有防御性设计，也就完全没有 harness。

> **【踩坑预警】** subprocess.run 的 returncode 被丢弃**
> **后果**：naive agent 的 `run_pytest` 函数只返回 `result.stdout`，而 `result.returncode` 被直接丢掉——这意味着即使 pytest 失败（exit code != 0），agent 拿到的也只是一段普通字符串，它会**以为任务成功**并继续在错误基础上操作，最终交付一份"通过了但实际没过"的烂代码。
> **正确做法**：显式保留 returncode 作为 passed 判定——`return {"passed": result.returncode == 0, "output": result.stdout + result.stderr}`，tool 返回必须含 status 字段（见 6.6 Verification Loop）。
> **排查方法**：检查任何 subprocess / shell tool 的返回值是否包含布尔成败标记；如果 agent 的 tool 返回只有 stdout 字符串，几乎可以肯定存在这类"成败信息丢失"的坑。

### 5.2 八大故障模式清单：这段代码将以八种方式失效

&emsp;&emsp;现在我们把上面代码里的 8 个故障模式整理成一张清单，你需要把它记住——**这张清单是本课剩下所有内容的"问题轴"**，第六章的八大机制每一个都对应这张清单里的某一行。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-1 naive agent 八大故障模式清单</font></p>
<div class="center">

| 故障模式 | 名称 | 现场表现 | 后果 |
|------|------|---------|------|
| ① | 循环失控（缺刹车机制） | `while True` 或无终止条件验证，模型以为任务没完就一直跑 | 消耗 API 额度、无限占用资源、可能失控 |
| ② | context 溢出 | `messages.append` 无限积累，轮次越多 prompt 越长 | 超 context window 直接报错；或 token 费用失控 |
| ③ | cache miss | system prompt 或 messages 前缀在循环内被改动 | 每轮 LLM 请求都无法命中 prompt cache，延迟翻倍 + 费用翻倍 |
| ④ | tool 错误吞 | `except Exception: result = ""` 或 subprocess returncode 被丢 | agent 以为成功但实际失败，继续在错误基础上操作 |
| ⑤ | 状态丢失 | 没有 progress 文件，session 一关全丢 | 长任务无法跨天续传，kill 进程后完全从零开始 |
| ⑥ | 缺权限闸（危险操作无拦截） | 无 Permission Gate，`rm -rf` / `sudo` / `git push --force` 能直接跑 | 数据毁损、生产事故 |
| ⑦ | 缺自动化评审（生成者自评偏见） | agent 自己说"做完了"但没有独立角色把关，质量无约束 | 你发现问题时产物已交付 |
| ⑧ | 成本失控 | 无 token budget，循环跑多久消耗多少成本 | 一个长任务可能耗尽你一个月的 API 预算 |

</div>

> 📌 **这张表是本课最重要的记忆载体**。你需要把它记到能 30 秒内背出来——因为第六章每讲一个机制都会说"它救的是这张表里的第 X 条"。双层锚定从这里开始。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260423171849459.png" width="50%">
</div>

### 5.3 八大故障模式 ↔ 八大机制

&emsp;&emsp;我们现在手里有一张问题矩阵（8 个故障模式），第六章会给我们一张解法矩阵（8 个机制）。这两张矩阵合起来就是 6.9 的机制×故障模式对照表——本课最核心的认知闭环。

> **【学习钩子】**：从下一节开始，每当看到一个新机制，先回头看这张故障模式表，问自己一句"它救的是哪一条？" 能对应上，你就真的学会了；对应不上，回来重看故障模式清单。

> **【学完本节你已经掌握】**
> - 记住**八大故障模式清单**：①循环失控 ②context 溢出 ③cache miss ④tool 错误吞 ⑤状态丢失 ⑥缺权限闸 ⑦缺自动化评审 ⑧成本失控
> - 能说出每一条的触发条件和后果（能 30 秒内背出来）
> - 能在任意 naive agent 代码中对照代码注释找到每个故障模式对应的行

---

## <center>第六章:驾驭工程的八大核心机制</center>

&emsp;&emsp;我们接下来进入本课最硬核的一段——八大核心机制。每个机制都用统一的四件套来讲：**一句话定义 + 官方出处 + 解哪条故障 + 生活化类比**，然后配一段 20-50 行的示意代码让你看到它的核心逻辑。

&emsp;&emsp;**关于代码示例**：每个机制的代码都是**骨架示意**，作用是让你一眼看清机制长啥样——你会看到 `call_llm / run_loop / TOOL_REGISTRY` 这样的占位或锚点注释。**这不是工程级实现**。完整可运行版本在**第二讲**，我们会在那里手搓一个 Mini Harness，把 8 大机制全部跑起来、看真实 token 消耗和 E2E 对比数据。

&emsp;&emsp;在展开之前我们先给出整体视图，把 8 个机制的地图摊开看清：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-0 八大机制总览</font></p>
<div class="center">

| # | 机制 | 一句话定义 | 解哪条故障 | 官方出处 |
|---|------|----------|-----------|---------|
| 6.1 | Agent Loop 四相循环 | Gather→Action→Verify→Iterate 带硬刹车 | ① | Claude Agent SDK（Anthropic 开源 Agent 开发 SDK，2025-09-29） |
| 6.2 | Tool Use 工具编排 | schema 契约让工具失败可被结构化回传 | ④ | Claude Agent SDK |
| 6.3 | Progress Tracking 进度追踪 | claude-progress.txt + git commit 跨 session 续传 | ⑤ | Anthropic Effective Harnesses（2025-11-26） |
| 6.4 | Context Management 上下文管理 | 监控 token + 阈值压缩/重置 | ②③ | Agent SDK + Compaction 文档 |
| 6.5 | Feature List 任务拆解 | JSON 清单 + 单次只做一件 | ②的源头 | Anthropic Effective Harnesses |
| 6.6 | Verification Loop 验证闭环 | Playwright/Puppeteer 真机验证 | ④的下游 | Anthropic Effective Harnesses |
| 6.7 | Subagents 子代理分治 | 独立 context 隔离的子任务分发 | ②的解法 | Claude Agent SDK |
| 6.8 | Generator-Evaluator（GAN 启发三角色） | Planner + Generator + Evaluator 分离 | ⑦ | Anthropic Harness Design（2026-03-24） |

</div>

&emsp;&emsp;注意看"解哪条故障"那一列——八大故障模式里的 **⑥缺权限闸 和 ⑧成本失控在本节不做主解**，它们会由第二节课的 **Permission Gate 权限闸 / Token Budget 成本预算** 两个扩展机制闭环。本节课 6.1-6.8 八大机制中，⑥在 Agent Loop（6.1）有部分覆盖，⑧在 Context Management（6.4）/ Feature List（6.5）/ Subagents（6.7）均有部分覆盖（详见 6.9 的机制×故障模式对照表）；但"主解"这一层的完整闭环要等到第二节课的 Permission Gate 和 Token Budget 才能交付。**本节我们先把其余 6 条（①②③④⑤⑦）全部解决掉**。

---

### 6.1 Agent Loop 四相循环（解 ① 循环失控）

&emsp;&emsp;第一个机制回答的问题最朴素——**怎么让一个循环不失控**。

**一句话定义**：把一次 agent 迭代拆成 Gather Context（收集上下文）→ Take Action（执行动作）→ Verify（验证结果）→ Iterate（迭代改进）四个阶段，并硬性加上 max_iterations 和显式终止条件。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260423171849433.svg" width="70%">
</div>

**官方出处**：[Building agents with the Claude Agent SDK](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)（Anthropic，2025-09-29，作者 Thariq Shihipar）。

**解哪条故障**：故障 ① 循环失控——naive agent 的 `while True` 在这里被替换成 `for i in range(MAX_ITER)`，每轮还有显式的 verify 阶段做"能不能停"的判断。

**生活化类比**：**驾校教练教你开车的四步口诀**——"看路况 → 踩油门 → 看前方 → 调整方向"。每一步都有明确动作，开完一圈看有没有到终点；到了就停、没到就再来一圈——但你不会无限圈兜下去，因为教练给你定了"跑 10 圈就必须停"的硬规则。

In [ ]:
# 机制 6.1：Agent Loop 四相循环
# 解 ① 号故障：带 max_iterations 硬刹车 + 显式终止条件

# —— 下列四个函数 + state.is_done() 是"四相"的教学标签 ——
# 在真实 agent 里对应到 cell [55] naive_agent 的具体片段：
#   gather_context(state) ≈ 构造/读取 messages（见 cell [55] messages 初始化）
#   take_action(context)  ≈ client.chat.completions.create(...)（见 cell [55] LLM 调用）
#   verify(action)        ≈ 机制 6.6 的 verify_with_pytest（真机验证）
#   iterate(state, r)     ≈ messages.append(...) + state 更新
#   state.is_done()       ≈ msg.tool_calls 为空时的 break 条件
MAX_ITER = 20  # 硬性上限，到点必停

def agent_loop(initial_state):
    """四相循环主函数
    
    Args:
        initial_state: 初始状态对象（含任务、历史、已完成标记）
    Returns:
        final_state: 循环结束时的最终状态
    """
    state = initial_state
    # for 循环代替 while True，保证最大迭代次数
    for i in range(MAX_ITER):
        context = gather_context(state)    # 相 1：收集上下文
        action = take_action(context)      # 相 2：执行动作
        result = verify(action)            # 相 3：验证结果
        state = iterate(state, result)     # 相 4：迭代状态
        # 显式终止条件：状态自报完成才退出
        if state.is_done():
            break
    return state

&emsp;&emsp;这段代码里最关键的一行是 `for i in range(MAX_ITER)` 取代了 naive agent 的 `while True`。我们可能会觉得"这不就是加了个计数器吗"——对，就是加了个计数器。但这个计数器背后的工程思维是：**agent 不该完全信任模型的自我终止判断，系统必须有一个强制出口**。Anthropic 官方博文里用的词是 "structured loop with guaranteed exit"——有保证的出口，这就是我们的第一道刹车。

&emsp;&emsp;**有了刹车之后，我们还面临另一个问题——工具失败了 agent 看不见。** 这就是第二个机制 Tool Use 要解决的。

---

### 6.2 Tool Use 工具编排（解 ④ tool 错误吞）

&emsp;&emsp;第二个机制回答的问题是——**工具失败了 agent 要看得见**。

**一句话定义**：所有工具调用包裹在结构化 schema 里（名字、参数、返回格式统一），失败时返回 `{"status": "error", "error": "..."}` 这样的结构化回传，**而不是空字符串或异常被吞**。

**官方出处**：[Building agents with the Claude Agent SDK](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)（Anthropic，2025-09-29）；MCP（Model Context Protocol，[Anthropic 2024-11-25 推出的开放协议](https://www.anthropic.com/news/model-context-protocol)，专门规定了 Agent 与工具之间的 schema 格式——你可以把它理解成：如果 Function Calling 是工具调用的接口约定，MCP 就是这个约定的行业标准化版，让任何 Agent 框架和任何工具都能说同一种语言；被称为 Agent 世界的 USB-C）官方规范把这种 schema 契约进一步标准化。

**解哪条故障**：故障 ④ tool 错误吞——naive agent 里 `except: result = ""` 是典型症状，修复方式是把 error 变成结构化 return，而不是字符串或异常。

**生活化类比**：**USB-C 接口**。任何设备插上 USB-C 线，host 都能通过标准协议读到对方的身份、能力、故障码；不兼容的设备也能返回"设备不识别"，而不是一条死线静默。工具 schema 就是 agent 世界的 USB-C。

In [ ]:
# 机制 6.2：Tool Use 工具编排
# 解 ④ 号故障：工具失败必须结构化回传，agent 才能感知

# TOOL_REGISTRY 示例：即 cell [55] 三个 tool 函数的字典封装，集中查表
TOOL_REGISTRY = {"read_file": read_file, "write_file": write_file, "run_pytest": run_pytest}


def safe_tool_call(tool_name: str, **kwargs) -> dict:
    """结构化工具调用包装器
    
    Args:
        tool_name: 工具名（从 TOOL_REGISTRY 查表）
        **kwargs: 工具参数（JSON schema 约束）
    Returns:
        dict: {"status": "ok"|"error", "result": 或 "error": 字符串}
    """
    try:
        # 成功路径：正常返回结果
        result = TOOL_REGISTRY[tool_name](**kwargs)
        return {"status": "ok", "result": result}
    except Exception as e:
        # 关键改进：error 不是空字符串，agent 能感知失败
        return {"status": "error", "error": str(e)}

&emsp;&emsp;对比 naive agent 里的 `except: result = ""` 和这段代码的 `return {"status": "error", "error": str(e)}`——两者差别只有一行，但工程含义完全不同。前者是"失败了装作没事"，后者是"失败了把失败本身当作第一公民返回"。agent 下一轮看到 `status: error` 才会触发修复路径，看到空字符串就以为成功。**结构化错误是 agent 能跑长任务的第一块基石**。

&emsp;&emsp;**工具失败能看见之后，我们还要解决一个更大的问题——进程一挂进度就全丢。** 这就是 Progress Tracking 要接手的地方。

---

### 6.3 Progress Tracking 进度追踪（解 ⑤ 状态丢失）

&emsp;&emsp;第三个机制回答的问题是——**kill 进程或者关电脑之后，agent 能不能从断点接着跑**。

**一句话定义**：把每一步的进度写到一个持久文件（Anthropic 官方用的是 `claude-progress.txt`）+ 在完成里程碑时打 git commit，这样即使 session 断了，重启时能读取进度文件从断点续传。

**官方出处**：[Effective Harnesses for Long-Running Agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)（Anthropic，2025-11-26，作者 Justin Young）。原文原话："a claude-progress.txt file that keeps a log of what agents have done"。

**解哪条故障**：故障 ⑤ 状态丢失——naive agent 里 session 关掉进度全丢，修复方式是把进度写磁盘 + 用 git 历史作为额外的 checkpoint。

**生活化类比**：**游戏存档**。开放世界游戏不可能指望玩家一次通关，必须每过一个小任务自动存档；agent 跑长任务也是同样——每完成一个子步骤就"存档"，断了能"读档"接着跑。

In [ ]:
# 机制 6.3：Progress Tracking
# 解 ⑤ 号故障：跨 session 续传的最小基础设施
import json
import pathlib

PROGRESS_FILE = pathlib.Path("claude-progress.json")


def save_progress(state: dict) -> None:
    """每步完成后写进度文件，失败后可续传
    
    Args:
        state: 当前 agent 状态（含已完成任务、上下文快照）
    """
    # 每步写磁盘，写 JSON 格式方便人类阅读和工具消费
    PROGRESS_FILE.write_text(json.dumps(state, indent=2, ensure_ascii=False))


def load_progress() -> dict | None:
    """启动时先读进度，有则续传
    
    Returns:
        dict 或 None：有进度则返回状态，无则返回 None（全新开始）
    """
    # 存在即续传，不存在即从零开始
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text())
    return None

&emsp;&emsp;这段代码看上去简单到不值得写成独立机制——不就是 JSON 读写吗？但我们需要想清楚一件事：**没有它，agent 跑 30 分钟失败一次就得从零开始**。Anthropic 官方博文给出的最佳实践更强：progress 文件 + git commit 双重 checkpoint——前者记录 agent 的思考过程，后者记录代码的物理改动，两个维度的断点都可以续上。

&emsp;&emsp;**有了进度持久化之后，另一个威胁迎面扑来——context window 本身在膨胀，迟早会溢出。** 这就是 Context Management 要管的事。

---

### 6.4 Context Management 上下文管理（解 ② context 溢出 / ③ cache miss）

&emsp;&emsp;第四个机制回答的问题是——**context window 要溢出怎么办**。

**一句话定义**：持续监控当前 messages 的 token 数，超过阈值时触发压缩（compaction）或重置（reset），同时**保持 system prompt 前缀稳定不变**以命中 prompt cache。

**官方出处**：[Automatic Context Compaction Cookbook](https://platform.claude.com/cookbook/tool-use-automatic-context-compaction)（Claude Agent SDK 官方）；Anthropic 在 2026 年 3 月前后的工程路线从"偏 compaction"逐步转向"偏 reset"。

**解哪条故障**：一次救两条——故障 ② context 溢出（压缩能降维度）和故障 ③ cache miss（保持前缀稳定才能命中 cache）。

**生活化类比**：**工位清理**。你工位上的文件堆到爬不动的时候，有两种策略：**压缩（把过期文件归档成一页摘要）** 或 **重置（今天新文件从空桌开始）**。agent 的 context 也一样，到某个阈值就得做出这个选择。

In [ ]:
# 机制 6.4：Context Management
# 解 ②③ 号故障：超阈值时压缩非前缀部分，保持 system prompt 前缀不变
CONTEXT_THRESHOLD = 80_000  # token 阈值（示意值，按模型实际 window 调整）


def call_llm_for_summary(msgs: list) -> str:
    """把一串 messages 压成一段摘要

    真实实现会调 LLM 生成带语义的总结；此处用最朴素的"拼接+截断"演示"压缩动作的形态"，
    让学员看清 messages 从 N 条 → 1 条 summary 的数据流，而不是 LLM 本身怎么总结。

    Args:
        msgs: 待压缩的消息列表（不含 system prompt）
    Returns:
        str: 摘要字符串
    """
    # 仅演示形态：把所有 content 拼接后截断；真实场景请换成 LLM 摘要
    joined = " | ".join(str(m.get("content", ""))[:80] for m in msgs)
    return f"[前 {len(msgs)} 条消息压缩摘要] {joined[:500]}"


def manage_context(messages: list, token_count: int) -> list:
    """上下文管理器：超阈值触发压缩

    Args:
        messages: 当前消息列表（OpenAI 格式）
        token_count: 当前总 token 数（由上层调用方估算）
    Returns:
        list: 处理后的 messages（未超阈值则原样返回）
    """
    # 未超阈值直接返回，保持 prompt cache 命中
    if token_count <= CONTEXT_THRESHOLD:
        return messages
    # 超阈值才触发压缩；system prompt（messages[0]）永远不动以保 cache
    summary = call_llm_for_summary(messages[1:])
    # 返回前缀稳定 + 压缩摘要的新 messages
    return [messages[0], {"role": "assistant", "content": summary}]

&emsp;&emsp;这段代码里最容易被忽略的是 `messages[0]` 这一行——**system prompt 永远不动**。为什么？因为 prompt cache 是按前缀匹配的，前缀一变整个 cache 作废，延迟翻倍、费用翻倍（解 ③ 号故障）。很多 Agent 开发者第一反应是"在 system prompt 里加时间戳或当前任务"，这是 cache miss 最常见的起因。正确做法是：system prompt 是恒定的"角色定义"，当前任务进 user 消息，永远不要动前缀。

> **【踩坑预警】** 在 system prompt 里加时间戳、用户名、当前任务**
> **后果**：很多开发者把当前任务状态、时间戳或 task_id 写进 system prompt，以为"让模型知道当前在做什么"更好——实际上这会让每轮的 system prompt 前缀都不同，prompt cache 全部失效，**延迟和费用双双翻倍**。在一个长任务里，这项失误的成本可能是数十美元。
> **正确做法**：system prompt 只放固定的"角色定义"（永不变），当前任务、时间、task_id 全部放进 user 消息。
> **排查方法**：看 API 返回里的 `cached_tokens` 字段——如果你在多轮循环中这个字段始终是 0 或远小于 system prompt 的 token 数，说明前缀被改动了、cache 没命中。

> **【常见误区】** 把 context window 当成"memory"**
> **后果**：很多初学者以为 "context 越大 agent 记得越多"，于是把所有历史对话都塞进 context 等同于"长期记忆"——实际上 context window 只是一次请求的**短期工作区**，超阈值必须清理；真正的跨 session 记忆由 6.3 Progress Tracking 这种持久化机制承担。
> **正确做法**：把 context window 当"工位"（本次会话要用的文件）、把 progress 文件当"档案柜"（跨 session 持久的记忆）；需要 agent 跨天记住的东西必须写进 progress 文件，不能指望 context。

&emsp;&emsp;**到这里我们已经看完了前 4 个机制——Agent Loop（救①）/ Tool Use（救④）/ Progress Tracking（救⑤）/ Context Management（救②③）。**

> 📌 **30 秒小结**：合上屏幕，不翻笔记，你能把这 4 个机制各解哪条故障说出来吗？说不清的回翻该节再过一遍；说得清的我们继续后 4 个机制。把 8 机制拆成 4+4 两组消化，认知负荷减半。

&emsp;&emsp;**context 在"量"上可以被清理，但源头上如果 agent 一次贪心做太多事，清理永远赶不上膨胀。** 于是我们需要 Feature List 把大任务从源头切细。

---

### 6.5 Feature List 任务拆解（解 ② context 溢出的源头）

&emsp;&emsp;第五个机制解决的是 context 溢出的**源头**——一次让 agent 做太多事。

**一句话定义**：把大任务拆成 JSON 格式的清单（`[{id, task, status}]`），**强制 agent 单次迭代只做一件事**，每件事做完标记状态再取下一件。

**官方出处**：[Effective Harnesses for Long-Running Agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)（Anthropic，2025-11-26）。原文给出了完整的 JSON feature 结构示例，passes 字段标注 true/false。

**解哪条故障**：故障 ② context 溢出的源头——naive agent 一次把 "读文件 + 修 bug + 跑 pytest" 全塞进去，context 自然爆。Feature List 把它切成 3 个独立子任务，每个子任务 context 小得多。

**生活化类比**：**待办清单**。你早上不会一口气处理 10 件事，而是列一个清单、做完一件划掉一件。agent 也一样——清单是它的**外置工作记忆**，不依赖 context window。

In [ ]:
# 机制 6.5：Feature List 任务拆解
# 解 ② 号故障的源头：强制单次只做一件事
import json

# 典型 feature list 结构：id / task / status 三字段
feature_list = [
    {"id": 1, "task": "读取 test_calculator.py", "status": "pending"},
    {"id": 2, "task": "修复 add 函数的整数溢出 bug", "status": "pending"},
    {"id": 3, "task": "运行 pytest 验证全部通过", "status": "pending"},
]


def get_next_task(features: list) -> dict | None:
    """取下一个 pending 任务，防止 agent 贪心多做
    
    Args:
        features: feature list（字典列表）
    Returns:
        dict 或 None：最早的 pending 任务，全部完成则 None
    """
    # 按 id 顺序取第一个 pending，强制单步执行
    # 这里用 next + 生成器是为了短路——找到第一个立即返回，不遍历全表
    # 如果 agent 想"一次做多件"，这个函数只返回一件，它就不得不分多轮
    return next((f for f in features if f["status"] == "pending"), None)

&emsp;&emsp;这个机制可以把它理解成 **agent 世界的 Trello 看板**。每个 task 是一张卡片、有明确的 pending / in_progress / passes 状态。Anthropic 官方建议这张清单以 JSON 文件形式存在磁盘上，而不是放在 context 里——这样它不会随 messages 一起被压缩掉。

&emsp;&emsp;**任务切细之后，每一件事做完了怎么确认它真的做对了？** 这就需要 Verification Loop 把"声称完成"和"真的完成"分开。

---

### 6.6 Verification Loop 验证闭环（解 ④ tool 错误的下游）

&emsp;&emsp;第六个机制跟 Tool Use（6.2）是一组搭档——6.2 管"工具层面的结构化错误"，6.6 管"任务层面的真机验证"。

**一句话定义**：每个 feature 完成后，用 Playwright / Puppeteer / pytest 等真机工具去**实际验证功能**，而不是只看 LLM 自己说"我觉得做好了"。验证失败就回退重做，验证通过才标记 passes=true。

**官方出处**：[Effective Harnesses for Long-Running Agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)（Anthropic，2025-11-26）。原文明确提到 Puppeteer MCP server 用于前端功能验证，以及一句非常狠的规则："It is unacceptable to remove or edit tests because this could lead to missing or buggy functionality"（**不可接受删掉或修改测试，因为这会导致功能缺失或带 bug**）。

**解哪条故障**：故障 ④ tool 错误吞的下游——光在工具调用层面返回 `error` 不够，任务层面必须有"真机验证"才能确认功能真的跑通。

**生活化类比**：**QA 验收测试**。开发说"我修好了"不算数，得 QA 真的点按钮、真的跑场景、真的截图对照才算数。

In [ ]:
# 机制 6.6：Verification Loop
# 解 ④ 号故障的下游：任务层面真机验证，不只看 LLM 自评
import subprocess


def verify_with_pytest() -> dict:
    """pytest 真机验证
    
    Returns:
        dict: {"passed": bool, "output": str}
    """
    # 真跑 pytest，不 mock 不 skip（mock 会让 agent 看到"假通过"）
    # --tb=short 让错误回溯简短，避免 stdout 溢出塞满 context
    result = subprocess.run(
        ["pytest", "--tb=short"], capture_output=True, text=True
    )
    # 关键：returncode 不丢弃，作为 passed 判定依据
    # stdout + stderr 合并返回，让 agent 能看到完整错误信息
    return {
        "passed": result.returncode == 0,
        "output": result.stdout + result.stderr,
    }

&emsp;&emsp;对比 naive agent 的 `run_pytest` 函数（它丢掉了 returncode），这里显式保留 `result.returncode == 0` 作为 passed 判定——**"通过"必须来自外部真实反馈，而不是 agent 自己说通过了**。Anthropic 官方还额外约束："不能改测试让它通过"——如果 agent 发现测试在报错，正确路径是"去修实现"，错误路径是"把测试改成能过"。这是保证验证闭环不被 agent 作弊的最后一道防线。

&emsp;&emsp;**但即便有了验证闭环，主 agent 仍然要处理大量中间证据，context 迟早会被"脏活"污染。** 这就是 Subagents 要提供的另一条隔离路径。

> 📌 **60 秒小结（6.5-6.6）**：Feature List 把任务从源头切细（救②源头），Verification Loop 把"声称完成"和"真的完成"分开（救④下游）。手里现在有 6 个机制——Agent Loop / Tool Use / Progress Tracking / Context Mgmt / Feature List / Verification Loop。深呼吸一下，还剩 Subagents（6.7）和 Generator-Evaluator（6.8）两个机制就完成八大机制的全景。

---

### 6.7 Subagents 子代理分治（解 ② context 溢出的另一种解法）

&emsp;&emsp;第七个机制是 context 溢出的**第二种解法**——和 Context Management（6.4）压缩、Feature List（6.5）拆分互补。

**一句话定义**：把特定子任务派发给一个**独立 context 的子 agent**，子 agent 在自己的 messages 列表里工作、完成后只把结果返回主 agent，主 agent 的 context 完全不受子任务中间过程污染。

**官方出处**：[Building agents with the Claude Agent SDK](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)（Anthropic，2025-09-29）。Claude Code 的 Task tool 本质就是 Subagent 实现（参见 [Effective Harnesses](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) 2025-11-26 关于 Subagents 章节）。

**解哪条故障**：故障 ② context 溢出的另一解法——主 agent context 保持干净，所有"脏活"（多轮试错、大量工具调用）都在子 agent 的独立 context 里消化。

**生活化类比**：**外包给顾问**。CEO 把一块专业问题外包给咨询顾问，顾问自己开了一个独立项目（独立资料库、独立讨论），几周后只交一份结论给 CEO。CEO 的主 context 没有被顾问讨论的中间文档污染。

In [ ]:
# 机制 6.7：Subagents 子代理分治
# 解 ② 号故障：独立 context 隔离，主 agent 不被子任务中间过程污染

# —— run_loop 即机制 6.1 的 agent_loop（完整四相循环） ——
# 重点不在 run_loop 内部怎么跑，而在 sub_messages 是全新列表，与主 agent 完全隔离：
# 主 agent 看不到子任务的中间 tool_calls 轨迹，只拿得到 subagent 的最终 return 值。
def subagent(task: str, tools: list) -> str:
    """子代理：独立 messages 列表完成子任务
    
    Args:
        task: 子任务描述（自然语言）
        tools: 子代理可用工具清单
    Returns:
        str: 子任务的最终结论（不含中间过程）
    """
    # 子代理有独立的 messages，不共享主 agent 历史
    sub_messages = [
        {"role": "system", "content": "你是专注于单一子任务的助手"},
        {"role": "user", "content": task},
    ]
    # 内部跑一个完整的 agent loop，但外部看不到中间步骤
    return run_loop(sub_messages, tools)

&emsp;&emsp;注意 `sub_messages = [...]` 这一行——它不是从主 agent 的 messages 复制过来的，而是**全新的独立列表**。这是 Subagents 机制的核心：**context 是隔离的**。主 agent 拿到的只有子 agent 的最终 return 值，中间子 agent 跑了 20 轮工具调用主 agent 一无所知——正因为"一无所知"，主 context 才能保持干净。

&emsp;&emsp;**分治解决了 context 隔离的问题，但还有一个更深层的质量问题——让同一个 LLM 既做生成又做评审，它会为自己辩护。** 这就需要我们借 GAN 的思路做架构级别的对抗。

---

### 6.8 Generator-Evaluator（GAN 启发的三角色架构，解 ⑦ 缺自动化评审）

&emsp;&emsp;最后一个机制是本课里最"有野心"的一个——**它不再是在单 agent 上打补丁，而是直接引入多 agent 架构**。

&emsp;&emsp;在展开之前我们先用一句话把 GAN 的核心讲清楚：**GAN（生成对抗网络）的核心思路是让两个神经网络互相博弈——一个负责生成、一个负责鉴别，通过对抗提高双方能力**。这里我们借用这个思路，但不是机器学习里的 GAN 训练过程，而是**两个 Agent 互相博弈的架构理念**——用独立角色之间的对抗代替单 agent 的自我反思。

**一句话定义**：借鉴生成对抗网络（GAN）的思路，把一个任务的处理流拆成**三个独立角色的 agent**——**Planner（规划者）拆任务**、**Generator（生成者）写代码**、**Evaluator（评价者）独立评审**。Evaluator 不看 Generator 的中间过程，只看最终产出；它以"怀疑者"视角打分，不通过就打回。

**官方出处**：[Harness Design for Long-Running Application Development](https://www.anthropic.com/engineering/harness-design-long-running-apps)（Anthropic，2026-03-24，作者 Prithvi Rajasekaran）。原文原话："Taking inspiration from Generative Adversarial Networks (GANs)"，并明确设计了 **three-agent architecture—planner, generator, and evaluator**。

**解哪条故障**：故障 ⑦ 缺自动化评审——naive agent 里没有任何独立评审角色，agent 自己说做完就做完；Generator-Evaluator 架构把"评审"作为一个独立 agent 机械执行的步骤，打破"生成者自评偏见"。

**生活化类比**：**论文评审制度**。作者（Generator）写论文，审稿人（Evaluator）不看作者的写作过程只看论文本身，按评分标准打分；编辑（Planner）在作者和审稿人之间做协调，决定修改方向和是否录用。三个角色分离才能保证论文质量，让作者自己评自己的论文质量无法保证。

In [ ]:
# 机制 6.8：Generator-Evaluator（GAN 启发的 Planner-Generator-Evaluator 三角色）
# 官方出处：Anthropic 2026-03-24 Prithvi Rajasekaran（https://www.anthropic.com/engineering/harness-design-long-running-apps）
# 解 ⑦ 号故障：自动化评审把关键质量门卡住

def call_llm(prompt: str) -> str:
    """LLM 调用 placeholder
    
    真实实现是 client.chat.completions.create(...)，见 cell [55] naive_agent 的 LLM 调用。
    此处用回显保证三角色的输入-输出形态可见，让学员聚焦三角色结构而非 LLM 细节。
    """
    return f"[LLM 响应占位] 针对 prompt 前 60 字：{prompt[:60]}... (真实场景由 LLM 生成)"


def planner(task: str) -> list:
    """规划者：把大任务拆成可评审的子任务清单
    
    Args:
        task: 原始任务描述
    Returns:
        list: 可评审的子任务清单
    """
    return call_llm(f"把任务拆成可独立评审的子任务清单：{task}")


def generator(subtask: str) -> str:
    """生成者：按规划逐项产出代码或解决方案
    
    Args:
        subtask: 单个子任务
    Returns:
        str: 生成的代码或方案
    """
    return call_llm(f"请完成以下子任务：{subtask}")


def evaluator(code: str, criteria: str) -> dict:
    """评价者：以怀疑者视角独立评审，不看生成过程
    
    Args:
        code: Generator 的产出（不含中间过程）
        criteria: 评审标准
    Returns:
        dict: {"approved": bool, "feedback": str}
    """
    # 不看生成过程，只看最终产出是否满足标准
    # 关键：prompt 里明确"批判性"，诱导 LLM 以审稿人角色挑刺而非辩护
    verdict = call_llm(f"批判性审查以下代码是否满足标准：{criteria}\n\n{code}")
    # 简化判定：verdict 含"通过"二字则认为 approved；工程代码应改为结构化输出（JSON）
    return {"approved": "通过" in verdict, "feedback": verdict}

&emsp;&emsp;注意 `evaluator` 的 docstring——"**不看生成过程**"。这是整个机制的精髓：Evaluator 只能看到最终产出，不能被 Generator 的"解释过程"污染。如果让同一个 LLM 既做生成又做评审，它会倾向于**为自己辩护**；但如果把评审单独抽成一个 agent，给它一个"怀疑者"角色的 prompt，它就会认真挑刺。**这就是 GAN 的核心——通过架构性约束制造对抗，而不是靠单 agent 的自我反思**。

&emsp;&emsp;走到这里，我们已经把八大机制的核心逻辑过了一遍。手里现在有：Agent Loop 的四相刹车（救①）/ Tool Use 的结构化错误（救④）/ Progress Tracking 的游戏存档（救⑤）/ Context Management 的工位清理（救②③）/ Feature List 的待办清单（救②源头）/ Verification Loop 的 QA 闸（救④下游）/ Subagents 的外包顾问（救②另一解法）/ Generator-Evaluator 的论文评审（救⑦）——这八个工具，就是你接下来看任何 Agent 产品时的解析框架。

> **【学完本节你已经掌握】**
> - 能说出八大机制的名称（Agent Loop / Tool Use / Progress Tracking / Context Mgmt / Feature List / Verification Loop / Subagents / Generator-Evaluator）
> - 能把每个机制对应到它救的故障编号（Agent Loop→① / Tool Use→④ / Progress Tracking→⑤ / Context Mgmt→②③ / Feature List→②源头 / Verification Loop→④下游 / Subagents→②另一解 / Gen-Eval→⑦）
> - 能读懂每个机制的示意代码核心逻辑

---

### 6.9 机制×故障模式对照表（本课第一张正交矩阵）

&emsp;&emsp;现在我们把八大机制和八大故障模式交叉起来，形成本课最重要的闭环表。**这张表是用来查的，不是用来背的**——等第八章归类练习时你会再用到它，现在先扫一遍建立索引感。

<p align="center"><font face="黑体" size=4>图 6-9 Harness 八大机制防线态势矩阵</font></p>
<div align="center">
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260423171702687.png" width="60%">

> **【第六章收束】**：表里的"部分（第二节补）"是刻意留的——**⑥缺权限闸** 和 **⑧成本失控** 在本节仅有部分覆盖，完整主解由第二节课的**第 9 机制（Permission Gate 权限闸）** 和**第 11 机制（Token Budget 成本预算）** 交付。本节课留这个缺口是刻意的——让你带着"还剩两条没解决"的认知走出第一节课，下节课才有新内容的动力。

&emsp;&emsp;走到这里，我们手里已经有一张 8×8 的正交矩阵。接下来会拿到**第二张正交视角**——从"支柱"角度重新切八大机制。

---

## <center>第七章:三支柱视角——看懂 Agent 产品的另一套坐标系</center>

&emsp;&emsp;**这一章大概花你 10 分钟**。假设你明天看到一个新的 Agent 产品发布——你怎么 3 分钟判断它能解决什么问题、它把工程投资主要押在哪儿？机制视角（第六章）告诉你它做了什么，但无法告诉你它的**工程投资重心**在哪。支柱视角正是用来补这个缺口——它给你一把从"工程维度"切 harness 的尺子。

&emsp;&emsp;前一节按"机制"切八个机制，这一节我们换一个视角——**按"工程维度"切**。这两种切法面向同一组机制，但回答的是不同的问题：机制视角回答"每个机制干什么"，支柱视角回答"它属于 harness 的哪个工程层面"。两个视角合起来，我们就有了一个**正交的坐标系**。

&emsp;&emsp;你可能会问：第六章已经有 8×8 的矩阵了，为什么还需要第二个坐标系？一句话回答——**机制视角是"清单式发现"，支柱视角是"重心式判断"**。机制视角能告诉你"某产品做了 Feature List + Subagents 这两个机制"，但只有支柱视角能告诉你"它把 70% 的工程投资押在 Context Engineering 这一柱上"。前者用来盘点，后者用来定位——当你做产品选型、看技术发布、或者评估自研 harness 时，要的是后者。

### 7.1 三支柱总图：Harness 的三个工程维度

&emsp;&emsp;三支柱是本课对 harness 的**工程维度抽象**——它不是某一位作者的原创概念，而是综合 Martin Fowler 系（Böckeler）、Anthropic Rajasekaran、Phil Schmid 三家博客里反复出现的工程主题，归纳出的三个**正交坐标轴**。

> 📌 **命名说明**：Garbage Collection 是本课程的工程类比命名，并非 Böckeler/Rajasekaran/Schmid 原文直接用词——三位作者讨论的是"压缩上下文 / 回收过期状态 / 控制成本"这一类主题，本课借编程语言里的 GC 概念把它们打包，便于和另两柱形成对称命名。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260423171849434.svg" width="70%">
</div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 7-1 三支柱核心问题与判断规则</font></p>
<div class="center">

| 支柱 | 核心问题 | 判断规则 |
|------|---------|---------|
| Context Engineering | Agent 能**看见**什么信息？看见的信息是否准确精简？ | 如果一个机制在"整理 agent 接下来输入的信息"，它属于这一柱 |
| Architectural Constraints | Agent 能**做**什么、不能做什么？动作空间在哪里有边界？ | 如果一个机制在"约束 agent 的能力或交互边界"，它属于这一柱 |
| Garbage Collection | Agent 能**跑多久**？成本和资源怎么回收？ | 如果一个机制在"清理过期状态 / 限制成本 / 回收资源"，它属于这一柱 |

</div>

> 📌 **来源说明**：三支柱不是单一作者的原创定义，是本课程综合 Martin Fowler 系博客（Böckeler 2026-04-02）、Anthropic Rajasekaran（2026-03-24）、Phil Schmid（2026-01-05）多来源抽象的**工程维度框架**。如果你在别的课程听到不一样的分法，不用奇怪——驾驭工程还在快速演化中，"三支柱"只是我们可以用的一种坐标系。

### 7.2 三支柱视角的价值：看一个新产品能立刻定位

&emsp;&emsp;**三支柱不是凭空的分类，而是能走通真实产品的工程坐标**。我们用 Claude Code 完整走一次三支柱归类，让"支柱"从定义变成可验证的判断：

- **Context Engineering (输入端：怎么在第一时间把对的资料喂给它)**：Claude Code 读取你项目目录里的 CLAUDE.md（相当于 RAG 系统 prompt）、在长会话时触发 session 压缩（compaction）、把进度写进 progress 文件——这三项都在"整理 agent 看见的信息"。

- **Architectural Constraints (控制流：它发疯的时候，你怎么硬生生把它按住)**：Claude Code 提供 27 种 hook 让你约束工具行为、内置 Permission Gate 控制危险操作、通过 Task tool（subagent 机制）做分治隔离——这三项都在"约束 agent 能做什么"（设红线、拉电闸）。

- **Garbage Collection (状态流：内存快被撑爆时，怎么有策略地遗忘)**：Claude Code 有 token budget 防爆预算、做 session compaction 回收过期上下文；Anthropic 2025→2026 的工程路线注释也在从 compaction 逐步转向 reset——这条支柱核心是容错与动态收敛。

三条推理走完，Claude Code 这个产品的"工程投资地图"就清晰了——**这就是支柱视角的定位能力**：不是取代机制视角，而是给你一个正交的第二轴，让任何产品都能被两个视角共同确认。

&emsp;&emsp;有了这个验证样例之后，再看抽象框架就更容易——**正交意味着我们获得了一个"定位能力"**。想象你在 4 月看到一个新 Agent 产品，可以立刻问两个问题：

- **按机制视角**：它解了八大故障模式里的哪几条？

- **按支柱视角**：它主要投资在哪一柱？

&emsp;&emsp;比如：CLI-first 产品（Claude Code / Codex CLI）在官方博文中强调**多机制组合**——既投 Context Engineering（compaction + progress file），也投 Architectural Constraints（Permission Gate + Subagents）。而 IDE-native、Cloud-sandbox、SDK-harness 家族，**具体每个产品的支柱主投资方向需结合各自官方文档判断**，不宜仅凭产品形态做结论。两个视角一套，产品定位立刻变得可讨论；但"投在哪一柱"这种判断不是本课的定论，而是你带着坐标系去读一手材料时的提问方式。

### 7.3 八大机制 × 三支柱对照表（本课第二张正交矩阵）

&emsp;&emsp;现在把第六章讲过的八个机制重新按三支柱分类——**同一组机制，换个视角看**。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 7-2 八大机制 × 三支柱归属</font></p>
<div class="center">

| 机制 | Context Engineering | Architectural Constraints | Garbage Collection | 归属判断说明 |
|------|--------------------|--------------------------|--------------------|------------|
| 6.1 Agent Loop 四相循环 | 辅 | **主** | - | 循环每轮都要整理 context；四相结构本身是架构约束 |
| 6.2 Tool Use 工具编排 | - | **主** | - | 工具 schema 本身就是一个动作空间的约束 |
| 6.3 Progress Tracking | **主** | 辅 | - | 跨 session 的状态持久化属于 context 延展 |
| 6.4 Context Management | **主** | - | 辅 | 本柱的核心；compaction 也有 GC 属性 |
| 6.5 Feature List | **主** | 辅 | 辅 | JSON 清单是 context 的规划层 |
| 6.6 Verification Loop | - | **主** | - | 事件驱动的错误显性化属于架构约束 |
| 6.7 Subagents | **主** | 辅 | 辅 | 分治隔离是架构性的边界划分 |
| 6.8 Generator-Evaluator | - | **主** | - | 三角色分离是架构约束的典型实现 |
| （待补）9 Permission Gate | - | **主** | - | 危险操作拦截属架构约束；第二节课补 |
| （待补）11 Token Budget | - | - | **主** | 成本控制属 GC；第二节课补 |

</div>

> 📌 **观察一件事**：表里 **Garbage Collection 那一列在第一节课基本是空的**——只有 Context Management 的"辅"。这不是漏讲了，而是**驾驭工程的 GC 支柱主要由 Token Budget / Cost Cap 这些机制承担**，它们是第二节课的核心内容。第一节课留这个空白是诚实的——告诉你"这一柱下节课补全"，而不是硬凑。

&emsp;&emsp;我们已经把八大机制从两个正交视角都看过一遍——**故障模式视角**（每条机制救哪个坑，表 6-9）+ **支柱视角**（每条机制属于哪个工程维度，表 7-2）。两张矩阵合起来就是你未来看任意 harness 产品的坐标系。

> **【学完本节你已经掌握】**
> - 能用 **三支柱坐标系**（Context Engineering / Architectural Constraints / Garbage Collection）定位任意 harness 机制
> - 能说出每个支柱回答的核心问题（看见什么 / 能做什么 / 能跑多久）
> - 能把八大机制主投资方向归到对应支柱上（见表 7-2）

---

## <center>第八章:生态地图——坐标系的实战运用</center>

&emsp;&emsp;学完机制和支柱，我们手里现在有一套坐标系。这一节我们用这套坐标系**实战**——看 10+ 个真实的 Harness 产品，用四系家族树把它们归类，然后做一个简短的归类练习检验你学到的东西。

### 8.1 10+ 主流 Harness 产品全景

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 8-1 10+ 主流 Harness 产品清单</font></p>
<div class="center">

| # | 产品 | 开发方 | 1 句定位 |
|---|------|-------|---------|
| 1 | Claude Code | Anthropic | CLI-first 驾驭工程参考实现，[Anthropic 官方博文](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)对其机制的文档化程度较为详尽（含 MCP / Subagent / Hook / Skill 等子系统） |
| 2 | Codex CLI | OpenAI | CLI-first，OpenAI 官方 harness 参考 |
| 3 | Cursor | Anysphere | IDE-native harness，VS Code 深度集成 |
| 4 | Windsurf | Codeium | IDE-native，IDE 端体验对标 Cursor |
| 5 | Cline | VS Code 社区 | IDE-native 开源插件，autonomous agent 特色 |
| 6 | Aider | 开源社区 | CLI-first 开源，git commit 深度集成 |
| 7 | Amp | Sourcegraph（2026-02-19 转 CLI-only） | CLI-first，代码搜索 + agent 结合 |
| 8 | Roo Code | 开源社区 | IDE-native VS Code 扩展，Cline 分支演化 |
| 9 | DeepAgents | LangChain | SDK-harness，基于 LangGraph 的 harness 组件库 |
| 10 | Devin | Cognition | Cloud-sandbox，完全托管的 autonomous agent |
| 11 | JetBrains Junie | JetBrains | IDE-native，JetBrains 系 IDE 集成 |

</div>

> 📌 **价格参考**：Devin 报价 $500/mo 是材料中记录的数据，具体最新价格待补充。Claude Code / Codex CLI / Cursor 等按订阅层级不同。

### 8.2 四系家族树：Harness 的四种形态

&emsp;&emsp;上面 11 个产品不是平铺的，它们其实可以归成四个"家族"——每个家族有**共同的形态特征**。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260423171849438.png" width="60%">
</div>

> 📌 **脚注说明**：Amp 于 2026-02-19 转 CLI-only；其余产品状态以各产品官网为准。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 8-2 四系家族特征对比</font></p>
<div class="center">

| 家族 | 主要特征 | 典型代表 | 适用场景 | 局限性 |
|------|---------|---------|---------|--------|
| CLI-first | 命令行为主入口，跑在本地 terminal | Claude Code / Codex CLI / Aider | 后端开发、系统工程、对 CLI 有强偏好的开发者 | 需要命令行习惯，无 GUI，非程序员门槛高 |
| IDE-native | IDE 内置集成，UI 优先 | Cursor / Windsurf / Cline / Junie | 前端开发、对 IDE 生态有依赖的开发者 | 重依赖特定 IDE 生态，跨 IDE 迁移成本高 |
| Cloud-sandbox | 云端完全托管，无需本地环境 | Devin | 团队协作、需要远程长任务托管 | 成本较高、数据隐私顾虑、本地文件系统访问受限 |
| SDK-harness | 提供 SDK 自建 harness | DeepAgents (LangChain) | 定制化场景、需要把 harness 嵌入自家产品 | 集成复杂度高，需要自行维护 harness 运行时 |

</div>

> **【学完本节你已经掌握】**
> - 能用 **四系家族树**（CLI-first / IDE-native / Cloud-sandbox / SDK-harness）归类任意新 Harness 产品
> - 拿到一张 **10+ 主流 Harness 产品的生态地图**（含开发方和一句定位）

---

## <center>第九章:三句官方警示</center>

&emsp;&emsp;最后我们做三件事——用官方警示收束"不要过度相信 harness"的工程理性；用一张"本讲带走清单"锁定你今天必须记住的 5 件事；用 3 道课后作业把学习延续到下节课之前。

### 9.1 三句官方警示：harness 不是银弹

&emsp;&emsp;课程讲到这里，你可能会有一种"harness 好像能解决一切"的错觉。本节我们用三句来自官方/权威的警示把这个错觉打掉。

> **【警示一】** harness 会过时**
> "A harness encodes assumptions that go stale. What works for Claude 3.5 may not work for Claude 4.5."
> ——Anthropic 官方博文（2025-11-26 [Effective Harnesses for Long-Running Agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)，Justin Young）
> 翻译：harness 固化了一组关于模型的假设，这些假设会过期。对 Claude 3.5 有效的 harness，换成 Claude 4.5 可能就失效了。
> **对我们的含义**：不要把 harness 当成一次性架构写死，要像管理软件一样持续迭代。
> **排查方法**：判断你的 harness 是否开始过时的信号——同样任务的成功率下降 5pp 以上、需要频繁手动干预、新模型版本上线后 benchmark 反而退步。出现任一条就该考虑裁剪 harness 而非继续加码。

> **【警示二】** Build to Delete（建造以便删除）**
> "The best harness is the one you're most willing to throw away and rewrite. The dataset of failures from your current harness is more valuable than the harness itself."
> ——Phil Schmid 博文 [2026-01-05](https://www.philschmid.de/agent-harness-2026)（Bitter Lesson 三原则之一）
> 翻译：最好的 harness 是**最愿意扔掉重写的那个**。当前 harness 的失败样本比 harness 本身更有价值。
> **对我们的含义**：不要给 harness 加舍不得删的复杂度。Manus 团队 6 个月重构 5 次的故事，本质就是这句话的实战演绎。
> **后果**：舍不得删 harness 的后果是——随模型能力提升，过时的刚性控制流反而降低成功率；Manus 五次重构的每一次，都是因为上一版"舍不得删的设计"成了下一版的瓶颈。

> **【警示三】** 三大开放争议还没定论**
> 驾驭工程作为新学科，**有三个核心设计选择社群仍在争论**，目前没有统一答案：
> - **多 Agent（Subagents 分治）vs 单 Loop（Feature List 拆分）**：Anthropic 和 OpenAI 偏前者，LangChain 部分产品偏后者。**推理层**：多 Agent 隔离性更好但协调开销更大（主 agent 要处理子 agent 的 return 协议、失败重试策略）；单 Loop 简单可控但 context 膨胀压力全在一个 loop 里——Anthropic 偏多 Agent 因为其 Task tool 天然支持，LangChain 的 Feature List 偏单 Loop 因为有 LangGraph checkpoint 做状态保障。
> - **Compaction（压缩旧 context）vs Reset（重置 context 重来）**：Anthropic 2025 年偏 compaction，2026 年 3 月前后开始偏 reset。**推理层**：compaction 保留历史上下文但引入摘要噪声（摘要本身可能丢失关键信息，且给 LLM 增加"去噪"负担）；reset 干净但丢失跨 session 推理链——所以 reset 方案强依赖 Progress Tracking 和 Feature List 做外置记忆补救。
> - **CLI-first vs IDE-native**：Claude Code / Codex CLI 路线 vs Cursor / Windsurf 路线，两种哲学尚未分出胜负。**推理层**：CLI-first 擅长"长任务自主运行 + 脚本化可编排"，IDE-native 擅长"行内交互式修改 + 视觉反馈"——两者面向不同的工作流场景。
> **对我们的含义**：如果有人告诉你"正确答案就是 X"，保持警惕——这门学科还在演化中。

### 9.2 本讲带走清单：5 件事必须记住

&emsp;&emsp;如果 120 分钟的内容只能带走 5 件事，就是下面这 5 件：

1. **一个公式**：Agent = Model + Harness（Vivek Trivedy 原创；Böckeler @ martinfowler.com 传播）。

2. **一张故障模式清单**：八大故障模式——①循环失控 ②context 溢出 ③cache miss ④tool 错误吞 ⑤状态丢失 ⑥缺权限闸 ⑦缺自动化评审 ⑧成本失控。

3. **一张机制清单**：八大机制——Agent Loop / Tool Use / Progress Tracking / Context Management / Feature List / Verification Loop / Subagents / Generator-Evaluator。

4. **一个坐标系**：三支柱正交视角——Context Engineering / Architectural Constraints / Garbage Collection。

5. **一张生态地图**：四系家族树——CLI-first / IDE-native / Cloud-sandbox / SDK-harness。

&emsp;&emsp;下节课预告：第二节课我们会补全 ⑥缺权限闸（Permission Gate 权限闸）和 ⑧成本失控（Token Budget 成本预算）这两个缺失的机制，并且会走进一个**真实的 harness 项目**——基于 DeepAgents 或 LangGraph 从零搭一个能跑 30 分钟长任务的 agent。

## <center>第十章:本讲小结</center>

&emsp;&emsp;120 分钟讲下来，你应该对"驾驭工程"这四个字有了完全不同于进课堂前的认知。它不是"feedback control 换皮"，它是一套被八大故障模式逼出来、由八大机制实现、在三支柱坐标系下工作、在 10+ 个真实产品上落地、但**注定会过时所以必须准备随时删除**的工程学科。

&emsp;&emsp;你进课堂时可能带着"这不就是换名字"的抵触；走出课堂时我们希望你带走的是三样东西：**一套坐标系**（八大故障模式 × 八大机制 × 三支柱）、**一张生态地图**（四系家族树 + 11 个产品）、以及 **一种工程理性**（harness 会过时 / Build to Delete / 三大开放争议）。

&emsp;&emsp;下节课我们会走进一个真实的 harness 项目，把今天这张坐标系套到代码上，看看 ①-⑧ 八大故障模式在一个工业项目里具体长什么样、八大机制每一条具体是怎么落地的。今天你记住的**故障模式清单、机制清单、三支柱**，就是下节课的"知识基底"。

&emsp;&emsp;课程到此结束。**如果你把课中的反思、课后的听讲、课后的必做阅读这三件事认真做完**，你现在就已经具备以下三项可以自测的能力——

---

> 📌 **本课信息源归属**：
> - Agent = Model + Harness 公式：Vivek Trivedy（[2025-09-23 首篇 HaaS 博文原创](https://vtrivedy.com/posts/claude-code-sdk-haas-harness-as-a-service)）/ Birgitta Böckeler @ [martinfowler.com](https://martinfowler.com/articles/harness-engineering.html)（2026-04-02 传播）
> - Terminal Bench +13.7pp 实证数据：LangChain 博客 2026-02-17，作者 Vivek Trivedy（[原文链接](https://www.langchain.com/blog/improving-deep-agents-with-harness-engineering)）
> - OpenAI 约 100 万行（on the order of）/ 零人工编码 / 1/10 时间：[OpenAI 工程团队博文 2026-02-11](https://openai.com/index/harness-engineering/)（核查通过，作者为团队署名）
> - 八大机制主要出处：[Claude Agent SDK 博文](https://www.anthropic.com/engineering/building-agents-with-the-claude-agent-sdk)（2025-09-29）+ [Anthropic Effective Harnesses](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)（2025-11-26，Justin Young）+ [Anthropic Harness Design](https://www.anthropic.com/engineering/harness-design-long-running-apps)（2026-03-24，Prithvi Rajasekaran）
> - 三支柱框架：本课程综合 Martin Fowler 系（Böckeler）/ Anthropic Rajasekaran / Phil Schmid 多来源的工程维度抽象
> - 术语首用节点：Vivek Trivedy（2025-09-23 博文 ["The Claude Code SDK and the Birth of HaaS"](https://vtrivedy.com/posts/claude-code-sdk-haas-harness-as-a-service)）
> 详细 HIGH 声明核查结果见同目录 `review/fact-check-phase1.5.json`。